In [1]:
import torch

import numpy as np
from skimage.transform import resize
from skimage.color import label2rgb
from PIL import Image
from PIL.ImageColor import getcolor
import matplotlib.pyplot as plt


from interactive_seg_backend import TrainingConfig, FeatureConfig, featurise, concat_feats, train_and_apply
from interactive_seg_backend.file_handling import load_labels
from dinosaw.models.vit_wrapper import PretrainedViTWrapper
from dinosaw.helpers import ModelTypes, get_models, get_features
from dinosaw.utils import do_2D_pca, closest_resize

SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = 'cuda:0'
half = False

2026-06-03 20:15:47 | I | multiscale_classical_cpu.py:  46 | N CPUS: 18


/home/ronan/Documents/phd/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
enabled_models: tuple[ModelTypes, ...] = ('dv2',)
model = get_models(enabled_models, "../../trained_models", DEVICE, half)['dv2']

bad_model = PretrainedViTWrapper("vit_small_patch14_dinov2.lvd142m", device=DEVICE, add_flash_attn=False)

f_cfg = FeatureConfig()
tr_cfg = TrainingConfig(feature_config=f_cfg, CRF=False, classifier='xgb', classifier_params={"class_weight": "balanced", "max_depth": 8}) 

In [3]:
sofc_img = Image.open("data/sofc.jpg").convert('RGB')
w, h = sofc_img.size

In [4]:
feats = get_features(model, sofc_img, device=DEVICE)
print(feats.shape)
feats_red = do_2D_pca(feats, 9, post_norm='minmax')
feats_red_hr = resize(feats_red, (h, w), order=1)
sofc_feats = feats_red_hr

(384, 64, 73)


In [5]:
labels = load_labels("data/sofc_labels.tiff")

pred, _, _ = train_and_apply(sofc_feats, labels, tr_cfg, image=np.array(sofc_img))

2026-06-03 20:15:56 | I | core.py                    : 155 | Training XGBClassifier: (10552, 9) -> ((10552,)) 
2026-06-03 20:15:56 | I | core.py                    : 170 | Applying XGBClassifier to (896, 1024, 9) features


/home/ronan/Documents/phd/.venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [20:15:56] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "class_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [6]:
COLOURS = [
    "#648FFF",
    "#785EF0",
    "#DC267F",
    "#FE6100",
    "#FFB000"
]
COLORS = [[v / 255.0 for v in getcolor(c, "RGB")] for c in COLOURS]

def hide_axis(ax):
    ax.set_xticks([])
    ax.set_yticks([])
    # ax.set_frame_on(False)

def apply_labels_as_overlay(labels: np.ndarray, img: Image.Image, colors: list, alpha: float=1.0) -> Image.Image:
    labels_unsqueezed = np.expand_dims(labels, -1)

    overlay = label2rgb(labels, colors=colors[1:], kind='overlay', bg_label=0, image_alpha=1, alpha=alpha)
    out = np.where(labels_unsqueezed, overlay * 255, np.array(img)).astype(np.uint8)
    img_with_labels = Image.fromarray(out)
    return img_with_labels

FS = 22

In [ ]:

plt.style.use("thesis.mplstyle")
H, W = 5, 5
fig, axs = plt.subplots(1, 3, figsize=(7, 2.3))

# plt.rcParams['font.family'] = 'serif' # or 'sans-serif' or 'monospace'
# plt.rcParams['font.serif'] = 'cmr10'
# plt.rcParams['font.sans-serif'] = 'cmss10'
# plt.rcParams['font.monospace'] = 'cmtt10'
# plt.rcParams["axes.formatter.use_mathtext"] = True # to fix the minus signs

plt.rcParams['text.usetex'] = True

img = apply_labels_as_overlay(labels, sofc_img, COLORS, alpha=1)

axs[0].imshow(img)
axs[1].imshow(feats_red_hr[:, :, :3])
axs[2].imshow(apply_labels_as_overlay(pred + 1, sofc_img, COLORS, alpha=1))

titles = ["Image with labels", "Features", "Predicted segmentation"]

for title, ax in zip(titles, axs):
    ax.set_title(title)
    hide_axis(ax)

ARROW_X = 1.1
ARROW_Y  = -0.1
axs[0].annotate(
    '', 
    xy=(0.05, ARROW_Y), 
    xycoords='axes fraction', 
    xytext=(3.4, ARROW_Y), 
    arrowprops=dict(
        arrowstyle="<|-",  # larger arrowhead
        color='black', 
        linewidth=1,
        shrinkA=0, 
        shrinkB=0,
        mutation_scale=10  # increase arrowhead size
    )
)
axs[0].text(ARROW_X + 0.6, ARROW_Y - 0.1, r'Train classifier (XGB) to map features $\rightarrow$ labels', transform=axs[0].transAxes,
             rotation=0, va='center', ha='center')

plt.savefig("out/bias_explained.pdf")
plt.close()